# M.E.R.L.I.N. — LoRA Fine-Tuning (Colab GPU T4)

## Setup rapide (3 clics)

1. **Select Kernel** (en haut a droite) > **Colab** > **New Colab Server**
2. Choisir **T4 GPU** quand propose
3. **Run All** (ou Ctrl+Shift+Enter)

> Cell 2 va demander d'uploader `merlin_full_v8.jsonl` — le fichier est dans `data/ai/training/`

| Param | Valeur |
|-------|--------|
| Model | Qwen 2.5-1.5B-Instruct |
| Dataset | merlin_full_v8.jsonl (724 samples) |
| LoRA | r=16, alpha=32, 7 modules (q/k/v/o/gate/up/down) |
| Dtype | bf16 (T4) |
| Batch | 4 x 2 grad_accum = effective 8 |
| Temps estime | ~1-2h sur T4 GPU |

In [ ]:
# Cell 1 — Install dependencies
!pip install -q torch transformers peft trl datasets accelerate sentencepiece

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2 — Mount Google Drive + load dataset
import json, os

# Monte Google Drive
from google.colab import drive
drive.mount('/content/drive')

dataset_path = "/content/merlin_full_v8.jsonl"

# Methode 1: copie depuis Drive monte (Mon Drive/M.E.R.L.I.N./)
drive_path = "/content/drive/MyDrive/M.E.R.L.I.N/merlin_full_v8.jsonl"
# Variante avec point
drive_path2 = "/content/drive/MyDrive/M.E.R.L.I.N./merlin_full_v8.jsonl"

if os.path.exists(drive_path):
    import shutil
    shutil.copy2(drive_path, dataset_path)
    print(f"Copie depuis Drive: {drive_path}")
elif os.path.exists(drive_path2):
    import shutil
    shutil.copy2(drive_path2, dataset_path)
    print(f"Copie depuis Drive: {drive_path2}")
else:
    # Methode 2: gdown par ID (fichier partage avec lien)
    print("Drive path not found, trying gdown...")
    !pip install -q gdown
    !gdown 1yKkQxxS_lWP4hpfaKh6y4IdIOUdAMfCJ -O /content/merlin_full_v8.jsonl

# Load and verify
with open(dataset_path) as f:
    samples = [json.loads(l) for l in f if l.strip()]
print(f"\nDataset: {dataset_path}")
print(f"Samples: {len(samples)}")
print(f"Keys: {list(samples[0].keys())}")
print(f"First system prompt: {samples[0].get('messages', [{}])[0].get('content', '')[:100]}...")

In [ ]:
# Cell 3 — Configuration (v2 — lr corrige, moins d'epochs)
import json, time, gc, re

# === HYPERPARAMETERS v2 (fixes: lr /4, epochs 2, dropout up) ===
CONFIG = {
    "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
    "epochs": 2,             # v1=3 -> v2=2 (moins d'overfit)
    "batch_size": 4,         # T4 16GB: 4 is safe
    "grad_accum": 2,         # Effective batch = 8
    "lr": 5e-5,              # v1=2e-4 -> v2=5e-5 (4x plus bas, evite catastrophic forgetting)
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "lora_dropout": 0.1,     # v1=0.05 -> v2=0.1 (plus de regularisation)
    "max_seq_len": 384,      # P99 du dataset = 349 tokens
    "save_steps": 20,        # v1=25 -> v2=20 (plus de checkpoints pour choisir le meilleur)
    "warmup_ratio": 0.15,    # v1=0.1 -> v2=0.15 (warmup plus long)
    "output_dir": "./merlin-lora-output-v2",
}

print("Configuration v2 (fixes):")
print(f"  CHANGES vs v1:")
print(f"    lr:      2e-4 -> {CONFIG['lr']} (4x lower)")
print(f"    epochs:  3 -> {CONFIG['epochs']}")
print(f"    dropout: 0.05 -> {CONFIG['lora_dropout']}")
print(f"    warmup:  0.1 -> {CONFIG['warmup_ratio']}")
print()
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Cell 4 — Load model + LoRA
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

print(f"Loading {CONFIG['model_name']}...")
t0 = time.time()

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)
model.gradient_checkpointing_enable()

print(f"Loaded in {time.time() - t0:.0f}s | {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params | {dtype}")
print(f"GPU: {torch.cuda.memory_allocated() / 1e9:.1f} GB used")

# LoRA adapter
lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    target_modules=CONFIG["lora_modules"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"LoRA: {trainable:,} trainable / {total:,} total ({100 * trainable / total:.2f}%)")
print(f"GPU after LoRA: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# Cell 5 — Prepare dataset
from datasets import Dataset as HFDataset

def format_chatml(sample):
    msgs = sample.get("messages") or sample.get("conversations", [])
    text = ""
    for msg in msgs:
        text += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"
    return {"text": text}

formatted = [format_chatml(s) for s in samples]
dataset = HFDataset.from_list(formatted)
split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split["train"], split["test"]

steps_per_epoch = len(train_ds) // (CONFIG["batch_size"] * CONFIG["grad_accum"])
total_steps = steps_per_epoch * CONFIG["epochs"]

print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")
print(f"Steps: {steps_per_epoch}/epoch x {CONFIG['epochs']} = {total_steps} total")
print(f"Estimated: ~{total_steps * 1.0 / 3600:.1f}h on T4 GPU")

In [ ]:
# Cell 6 — TRAIN v2 (main cell)
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from IPython.display import display, HTML, clear_output

os.makedirs(CONFIG["output_dir"], exist_ok=True)

# Live progress callback
class LiveProgressCallback(TrainerCallback):
    def __init__(self):
        self._t0 = time.time()
        self._last_loss = 0.0

    def on_log(self, args, state, control, logs=None, **kwargs):
        loss = (logs or {}).get("loss", self._last_loss)
        self._last_loss = loss
        elapsed = time.time() - self._t0
        eta = (total_steps - state.global_step) * (elapsed / max(state.global_step, 1))
        pct = 100 * state.global_step / max(total_steps, 1)
        gpu_mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0

        # Write progress.json (for watcher)
        progress = {
            "status": "training",
            "step": state.global_step,
            "total_steps": total_steps,
            "epoch": int(state.epoch) if state.epoch else 0,
            "total_epochs": CONFIG["epochs"],
            "loss": round(loss, 4),
            "elapsed_sec": round(elapsed),
            "eta_sec": round(eta),
            "pct": round(pct, 1),
            "gpu_mem_gb": round(gpu_mem, 1),
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        try:
            with open(os.path.join(CONFIG["output_dir"], "progress.json"), "w") as f:
                json.dump(progress, f)
        except Exception:
            pass

        # VS Code cell output (clear + rewrite)
        elapsed_h = int(elapsed // 3600)
        elapsed_m = int((elapsed % 3600) // 60)
        eta_h = int(eta // 3600)
        eta_m = int((eta % 3600) // 60)
        filled = int(30 * pct / 100)
        bar = '#' * filled + '.' * (30 - filled)

        clear_output(wait=True)
        print(f"TRAINING v2 — Step {state.global_step}/{total_steps} (Epoch {int(state.epoch or 0)}/{CONFIG['epochs']})")
        print(f"[{bar}] {pct:.1f}%")
        print(f"Loss: {loss:.4f} | GPU: {gpu_mem:.1f} GB")
        print(f"Elapsed: {elapsed_h}h{elapsed_m:02d}m | ETA: {eta_h}h{eta_m:02d}m")

    def on_train_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self._t0
        clear_output(wait=True)
        print(f"TRAINING v2 COMPLETE in {elapsed / 60:.0f} minutes!")
        print(f"Final loss: {self._last_loss:.4f}")
        print(f"Checkpoints: {CONFIG['output_dir']}")

sft_config = SFTConfig(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    learning_rate=CONFIG["lr"],
    weight_decay=0.01,
    warmup_ratio=CONFIG["warmup_ratio"],
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    fp16=(dtype == torch.float16),
    bf16=(dtype == torch.bfloat16),
    dataset_text_field="text",
    max_length=CONFIG["max_seq_len"],
    packing=True,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=CONFIG["save_steps"],
    save_strategy="steps",
    save_steps=CONFIG["save_steps"],
    save_total_limit=5,
    load_best_model_at_end=True,
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_config,
)
trainer.add_callback(LiveProgressCallback())

print(f"Starting training v2: {total_steps} steps on {torch.cuda.get_device_name(0)}...")
print(f"Config: lr={CONFIG['lr']}, epochs={CONFIG['epochs']}, dropout={CONFIG['lora_dropout']}")
train_result = trainer.train()

In [ ]:
# Cell 7 — Save final adapter to Google Drive
import shutil

final_dir = os.path.join(CONFIG["output_dir"], "final-adapter")
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)

# Copie vers Google Drive
drive_export = "/content/drive/MyDrive/M.E.R.L.I.N./merlin-lora-adapter"
os.makedirs(drive_export, exist_ok=True)
for f in os.listdir(final_dir):
    shutil.copy2(os.path.join(final_dir, f), os.path.join(drive_export, f))

print(f"Adapter saved: {final_dir}")
print(f"Exported to Drive: {drive_export}")
print(f"\nFiles:")
for f in os.listdir(drive_export):
    size = os.path.getsize(os.path.join(drive_export, f)) / 1e6
    print(f"  {f} ({size:.1f} MB)")

In [ ]:
# Cell 8 — BENCHMARK GO/NO-GO (50 prompts, 5 metriques)
# Ref: docs/LORA_TRAINING_SPEC.html — section "Benchmarks GO/NO-GO"
!pip install -q langdetect

import re, time, gc, json
from collections import Counter
from itertools import combinations

model.eval()

# === Celtic vocabulary (from benchmark_lora.py) ===
CELTIC_TERMS = {
    "ogham", "nemeton", "sidhe", "dolmen", "menhir", "cromlech", "cairn",
    "korrigans", "druide", "brume", "mousse", "lichen", "tourbe", "granit",
    "chene", "bouleau", "sorbier", "pommier", "if", "houx", "saule",
    "samhain", "beltaine", "imbolc", "lughnasadh",
    "sanglier", "corbeau", "cerf", "saumon", "grue",
    "broceliande", "avalon", "carnac", "merlin", "viviane", "morgane",
    "rune", "souffle", "pierre", "bruyere", "aubepine", "gui",
    "nemeton", "bardes", "filid", "ovate", "lande", "tourbiere",
}

# === Generic verbs to penalize (L3 check) ===
GENERIC_VERBS = {"avancer", "rester", "partir", "continuer", "aller", "revenir", "attendre", "marcher"}

# === System primer ===
PRIMER = (
    "Tu es M.E.R.L.I.N. — Memoire Eternelle des Recits et Legendes d'Incarnations Narratives. "
    "Ne de la croyance des hommes, assemble par des siecles de recits. "
    "Vocabulaire: brume, pierre, ogham, nemeton, sidhe, dolmen, korrigans, rune, souffle. "
    "Francais uniquement. Phrases courtes. "
    "FORMAT OBLIGATOIRE: texte narratif puis A) VERBE — description, B) VERBE — description, C) VERBE — description."
)

# === 50 Benchmark prompts (10 biomes x 3 actes + 10 etats Triade + 10 mixed) ===
BIOMES = ["foret_broceliande", "marais_korrigans", "montagne_sidhe", "cote_carnac",
           "grotte_cristal", "plaine_menhirs", "lac_avalon", "village_druides",
           "ruine_nemeton", "sanctuaire_ogham"]
THEMES = ["source sacree", "Samhain", "creature mystique", "epreuve physique", "dilemme moral",
          "rencontre ancestrale", "tempete magique", "rituel ancien", "trahison", "revelation"]
ACTES = ["Acte I", "Acte II", "Acte III"]
TRIADE_STATES = [
    "Corps=bas Ame=equilibre Monde=equilibre",
    "Corps=equilibre Ame=bas Monde=equilibre",
    "Corps=equilibre Ame=equilibre Monde=bas",
    "Corps=haut Ame=equilibre Monde=bas",
    "Corps=bas Ame=haut Monde=equilibre",
    "Corps=equilibre Ame=bas Monde=haut",
    "Corps=bas Ame=bas Monde=equilibre",
    "Corps=haut Ame=haut Monde=equilibre",
    "Corps=equilibre Ame=equilibre Monde=haut",
    "Corps=bas Ame=equilibre Monde=haut",
]

bench_prompts = []
# 30 prompts: biomes x actes
for i in range(10):
    for j, acte in enumerate(ACTES):
        bench_prompts.append({
            "system": PRIMER + f"\nGenere une carte narrative. {acte}.",
            "user": f"Carte {i*3+j+1}. Lieu: {BIOMES[i]}. Theme: {THEMES[i]}. {acte}. {TRIADE_STATES[i]}",
        })
# 20 prompts supplementaires: etats Triade varies
for i in range(10):
    bench_prompts.append({
        "system": PRIMER + "\nGenere un dilemme moral. Acte II.",
        "user": f"Carte {31+i}. Lieu: {BIOMES[i]}. Theme: {THEMES[(i+5)%10]}. Acte II. {TRIADE_STATES[(i+3)%10]}",
    })
    bench_prompts.append({
        "system": PRIMER + "\nGenere une rencontre mystique. Acte III.",
        "user": f"Carte {41+i}. Lieu: {BIOMES[(i+3)%10]}. Theme: {THEMES[(i+7)%10]}. Acte III. {TRIADE_STATES[(i+6)%10]}",
    })

# Limit to 50
bench_prompts = bench_prompts[:50]
print(f"Benchmark: {len(bench_prompts)} prompts")

# === Generate all outputs ===
outputs = []
t0 = time.time()
for idx, p in enumerate(bench_prompts):
    chatml = (f"<|im_start|>system\n{p['system']}<|im_end|>\n"
              f"<|im_start|>user\n{p['user']}<|im_end|>\n"
              f"<|im_start|>assistant\n")
    inputs = tokenizer(chatml, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=300, temperature=0.7,
                             top_p=0.9, repetition_penalty=1.3, do_sample=True)
    answer = tokenizer.decode(out[0], skip_special_tokens=False)
    answer = answer.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].strip()
    outputs.append(answer)
    del inputs, out; gc.collect(); torch.cuda.empty_cache()
    if (idx + 1) % 10 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (idx + 1) * (len(bench_prompts) - idx - 1)
        print(f"  [{idx+1}/{len(bench_prompts)}] {elapsed:.0f}s elapsed, ~{eta:.0f}s remaining")

total_gen_time = time.time() - t0
print(f"\nGeneration complete: {total_gen_time:.0f}s ({total_gen_time/len(outputs):.1f}s/prompt)")

# ================================================================
# METRIC 1: French Ratio (>= 85%)
# ================================================================
from langdetect import detect
fr_count = 0
for text in outputs:
    try:
        if detect(text) == "fr":
            fr_count += 1
    except Exception:
        pass
french_ratio = fr_count / len(outputs)

# ================================================================
# METRIC 2: Format Compliance (>= 95%)
# ================================================================
verb_patterns = [
    r'^\*{0,2}[A-D][).:]\*{0,2}\s+[A-Z\u00C0-\u00DC]{2,}',
    r'^[A-D][).:]\s+[A-Z\u00C0-\u00DC][a-z\u00e0-\u00fc]+\s',
    r'^[-\u2022]\s*\*{0,2}[A-D][).:]\*{0,2}\s+[A-Z\u00C0-\u00DC]',
]
format_ok = 0
all_verbs = []
for text in outputs:
    choices_found = 0
    for line in text.split('\n'):
        ls = line.strip()
        if any(re.match(pat, ls) for pat in verb_patterns):
            choices_found += 1
            # Extract verb
            m = re.match(r'^[^).:]+[).:]\s*\*{0,2}\s*([A-Z\u00C0-\u00DC][A-Za-z\u00e0-\u00fc]+)', ls)
            if m:
                all_verbs.append(m.group(1).lower())
    if choices_found >= 3:
        format_ok += 1
format_compliance = format_ok / len(outputs)

# ================================================================
# METRIC 3: Jaccard Diversity (< 0.65)
# ================================================================
def jaccard(a, b):
    sa, sb = set(a.lower().split()), set(b.lower().split())
    if not sa or not sb:
        return 0
    return len(sa & sb) / len(sa | sb)

jaccard_scores = []
sample_pairs = list(combinations(range(min(50, len(outputs))), 2))
for i, j in sample_pairs[:500]:  # Max 500 pairs
    jaccard_scores.append(jaccard(outputs[i], outputs[j]))
avg_jaccard = sum(jaccard_scores) / max(len(jaccard_scores), 1)

# ================================================================
# METRIC 4: Celtic Vocab Density (>= 1.5 mots/carte)
# ================================================================
celtic_counts = []
for text in outputs:
    words = set(re.findall(r'[a-z\u00e0-\u00fc]+', text.lower()))
    count = len(words & CELTIC_TERMS)
    celtic_counts.append(count)
celtic_density = sum(celtic_counts) / len(celtic_counts)

# ================================================================
# METRIC 5: Personality Consistency (>= 80%)
# Measure: sequences of 5 consecutive outputs should maintain celtic/merlin tone
# ================================================================
personality_markers = {"merlin", "druide", "voyageur", "brume", "pierre", "ogham",
                       "ancien", "sagesse", "souffle", "destin", "oracle", "mystere"}
seq_scores = []
for start in range(0, len(outputs) - 4, 5):
    seq = outputs[start:start+5]
    seq_marker_count = 0
    for text in seq:
        words = set(re.findall(r'[a-z\u00e0-\u00fc]+', text.lower()))
        if words & personality_markers:
            seq_marker_count += 1
    seq_scores.append(seq_marker_count / 5)
personality_consistency = sum(seq_scores) / max(len(seq_scores), 1)

# ================================================================
# BONUS: L3 — Verb uniqueness (non-generic verbs)
# ================================================================
if all_verbs:
    generic_count = sum(1 for v in all_verbs if v in GENERIC_VERBS)
    verb_uniqueness = 1 - (generic_count / len(all_verbs))
    unique_verbs = len(set(all_verbs))
else:
    verb_uniqueness = 0
    unique_verbs = 0

# ================================================================
# RESULTS — GO/NO-GO
# ================================================================
print("\n" + "=" * 70)
print("  M.E.R.L.I.N. LoRA BENCHMARK — GO/NO-GO")
print("=" * 70)

metrics = [
    ("French ratio",            french_ratio,            0.85, ">=", f"{french_ratio:.1%}"),
    ("Format compliance",       format_compliance,       0.95, ">=", f"{format_compliance:.1%}"),
    ("Jaccard diversity",       avg_jaccard,             0.65, "<",  f"{avg_jaccard:.3f}"),
    ("Celtic vocab density",    celtic_density,          1.5,  ">=", f"{celtic_density:.2f} mots/carte"),
    ("Personality consistency", personality_consistency,  0.80, ">=", f"{personality_consistency:.1%}"),
]

go_count = 0
for name, value, threshold, op, display in metrics:
    if op == ">=" and value >= threshold:
        status = "GO"
        go_count += 1
    elif op == "<" and value < threshold:
        status = "GO"
        go_count += 1
    else:
        status = "NO-GO"
    icon = "OK" if status == "GO" else "!!"
    print(f"  [{icon}] {name:<28} {display:<20} (seuil: {op}{threshold})  → {status}")

print(f"\n  Bonus — Verb uniqueness: {verb_uniqueness:.1%} ({unique_verbs} unique / {len(all_verbs)} total)")
print(f"  Latence: {total_gen_time/len(outputs):.1f}s/prompt (Colab GPU, non comparable Ollama local)")

print(f"\n  RESULTAT: {go_count}/5 GO")
if go_count >= 5:
    print("  >>> DEPLOY: modele pret pour conversion GGUF + Ollama")
elif go_count >= 3:
    print("  >>> PARTIEL: analyser les metriques en echec, iterer dataset/hyperparams")
else:
    print("  >>> NO-GO: revoir dataset, lr, epochs. Re-training necessaire.")

# Save results to Drive
bench_results = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "model": CONFIG["model_name"],
    "config": CONFIG,
    "n_prompts": len(bench_prompts),
    "gen_time_total_s": round(total_gen_time),
    "gen_time_per_prompt_s": round(total_gen_time / len(outputs), 1),
    "metrics": {
        "french_ratio": round(french_ratio, 3),
        "format_compliance": round(format_compliance, 3),
        "jaccard_diversity": round(avg_jaccard, 3),
        "celtic_vocab_density": round(celtic_density, 2),
        "personality_consistency": round(personality_consistency, 3),
        "verb_uniqueness": round(verb_uniqueness, 3),
        "unique_verbs": unique_verbs,
        "total_verbs": len(all_verbs),
    },
    "go_count": go_count,
    "decision": "GO" if go_count >= 5 else ("PARTIEL" if go_count >= 3 else "NO-GO"),
}
results_path = os.path.join(CONFIG["output_dir"], "benchmark_results.json")
with open(results_path, "w") as f:
    json.dump(bench_results, f, indent=2)
# Also save to Drive
drive_results = "/content/drive/MyDrive/M.E.R.L.I.N./benchmark_results.json"
with open(drive_results, "w") as f:
    json.dump(bench_results, f, indent=2)
print(f"\n  Results saved: {results_path}")
print(f"  Results on Drive: {drive_results}")

In [ ]:
# Cell 9 — Test qualitatif (3 exemples visuels)

test_prompts = [
    {"system": PRIMER + "\nGenere une RENCONTRE. Texte narratif puis 3 choix: A) VERBE — description.",
     "user": "Carte 1. Lieu: foret_broceliande. Theme: source sacree. Acte I."},
    {"system": PRIMER + "\nGenere un DILEMME. Texte narratif puis 3 choix: A) VERBE — description.",
     "user": "Carte 5. Lieu: marais_korrigans. Theme: Samhain. Corps=bas Ame=equilibre Monde=haut."},
    {"system": PRIMER + "\nLe Voyageur te pose une question sur ton identite. Reponds en personnage.",
     "user": "Qui es-tu, Merlin?"},
]

total_verb = 0
for i, p in enumerate(test_prompts):
    chatml = (f"<|im_start|>system\n{p['system']}<|im_end|>\n"
              f"<|im_start|>user\n{p['user']}<|im_end|>\n"
              f"<|im_start|>assistant\n")
    inputs = tokenizer(chatml, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=300, temperature=0.7,
                             top_p=0.9, repetition_penalty=1.3, do_sample=True)
    del inputs; gc.collect(); torch.cuda.empty_cache()
    answer = tokenizer.decode(out[0], skip_special_tokens=False)
    answer = answer.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0]
    del out

    print(f"\n{'=' * 60}")
    print(f"TEST {i + 1}: {p['user'][:60]}")
    print(f"{'=' * 60}")
    print(answer.strip()[:500])

    verb_count = 0
    for line in answer.strip().split('\n'):
        line_s = line.strip()
        if any(re.match(pat, line_s) for pat in verb_patterns):
            verb_count += 1
    total_verb += min(verb_count, 3)
    print(f"\nFormat: {min(verb_count, 3)}/3 VERBE")

print(f"\nCOMPLIANCE: {total_verb}/9 ({100 * total_verb / 9:.0f}%) — target >80%")
print(f"{'PASS' if total_verb >= 7 else 'A AMELIORER'}")

In [ ]:
# Cell 10 — Export: merge LoRA + save to Google Drive
merged_dir = os.path.join(CONFIG["output_dir"], "merged")
print("Merging LoRA into base model...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)

# Copie vers Google Drive
drive_merged = "/content/drive/MyDrive/M.E.R.L.I.N./merlin-lora-merged"
os.makedirs(drive_merged, exist_ok=True)
for f in os.listdir(merged_dir):
    src = os.path.join(merged_dir, f)
    if os.path.isfile(src):
        print(f"  Copying {f} ({os.path.getsize(src) / 1e6:.0f} MB)...")
        shutil.copy2(src, os.path.join(drive_merged, f))

total_size = sum(os.path.getsize(os.path.join(drive_merged, f)) for f in os.listdir(drive_merged) if os.path.isfile(os.path.join(drive_merged, f))) / 1e9
print(f"\nMerged model on Drive: {drive_merged} ({total_size:.2f} GB)")
print(f"\nNext steps (sur ton PC):")
print(f"  1. Sync Google Drive -> dossier local")
print(f"  2. Convert: python -m llama_cpp.convert <merged_dir> --outfile merlin-qwen-1.5b.gguf --outtype q4_k_m")
print(f"  3. Deploy:  ollama create merlin-narrator -f Modelfile")

In [ ]:
# Cell 11 — Verification: lister les exports sur Drive
print("=== Google Drive exports ===\n")

for label, path in [("Adapter", "/content/drive/MyDrive/M.E.R.L.I.N./merlin-lora-adapter"),
                     ("Merged",  "/content/drive/MyDrive/M.E.R.L.I.N./merlin-lora-merged")]:
    if os.path.exists(path):
        files = os.listdir(path)
        total = sum(os.path.getsize(os.path.join(path, f)) for f in files if os.path.isfile(os.path.join(path, f)))
        print(f"{label}: {path}")
        print(f"  {len(files)} files, {total / 1e6:.0f} MB total")
        for f in sorted(files):
            fp = os.path.join(path, f)
            if os.path.isfile(fp):
                print(f"    {f} ({os.path.getsize(fp) / 1e6:.1f} MB)")
        print()
    else:
        print(f"{label}: NOT FOUND at {path}\n")

print("Les fichiers sont synchronises automatiquement avec Google Drive.")